# RUKOPYS Qwen3-VL Stage 2: Gold Fine-tune (Hybrid Prompt v2)


In [ ]:
# Kaggle dependency cell.
# If your Kaggle image already has recent packages, set INSTALL_DEPS = False.
INSTALL_DEPS = True

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "accelerate",
            "peft",
            "bitsandbytes",
            "trl",
            "qwen-vl-utils",
            "datasets",
            "pandas==2.2.2",
            "pillow<12",
        ],
        [sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/transformers.git"],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd))
        subprocess.check_call(cmd)


In [ ]:
import gc
import json
import math
import os
import random
import re
import time
from collections import defaultdict
from pathlib import Path

import torch
from datasets import Dataset
from PIL import Image

Image.MAX_IMAGE_PIXELS = None
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

BASE_MODEL_CANDIDATES = [
    "/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1",
    "/kaggle/input/qwen3-vl-8b-instruct",
    "Qwen/Qwen3-VL-8B-Instruct",
]

DATASET_ROOT = "/kaggle/input/datasets/quii29/rukopys-dataset"

# Prefer the Stage 1 adapter trained with the same hybrid_prompt_v2 instructions.
# Add the Stage 1 notebook output as a Kaggle input, or run Stage 2 in the same session after Stage 1.
STAGE1_LORA_CANDIDATES = [
    "/kaggle/working/qwen3vl_rukopys_stage1_silver_hybrid_prompt_v2/qwen3vl_silver_lora_final",
    "/kaggle/input/qwen3vl-rukopys-stage1-silver-hybrid-prompt-v2/qwen3vl_silver_lora_final",
    "/kaggle/input/qwen3vl-rukopys-stage1-silver-prompt-v2/qwen3vl_silver_lora_final",
    "/kaggle/input/datasets/lhongthyan/htd-fine-tune-dataset/qwen3vl_rukopys_stage1_silver/qwen3vl_silver_lora_final",
]

OUTPUT_DIR = Path("/kaggle/working/qwen3vl_rukopys_stage2_gold_hybrid_prompt_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_SILVER = False
SILVER_SPLIT = "silver"
VAL_RATIO = 0.12

STAGE2_GOLD_MAX_STEPS = 1400
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
DATALOADER_NUM_WORKERS = 2
MODEL_DEVICE_MAP = "balanced"

GOLD_PAGE_LIMIT = 0
GOLD_CROP_LIMIT = 20000

MAX_SEQ_LENGTH = 3072
MAX_PIXELS_PAGE = 850_000
MAX_PIXELS_CROP = 320_000

MAX_PAGE_REGIONS = 60
MAX_PAGE_ANSWER_CHARS = 9000
MIN_BOX_SIDE = 8
CROP_PAD_RATIO = 0.035

PROMPT_VERSION = "hybrid_prompt_v2"

PAGE_SPECIAL_TEXT_MARKER_RULES = (
    "Use these special markers inside text fields only when they are visible in the region: "
    "~~word~~ for strikethrough text, ~~old~~{new} for strikethrough text with a visible correction, "
    "and [illegible] for an unreadable word inside an otherwise legible line. "
)

SPECIAL_TEXT_MARKER_RULES = (
    "Use these special markers only when they are visible in the crop: "
    "~~word~~ for strikethrough text, ~~old~~{new} for strikethrough text with a visible correction, "
    "and [illegible] for an unreadable word inside an otherwise legible line. "
)

DOCUMENT_CONTEXT_RULES = (
    "The crop may come from Ukrainian dictation handwriting, historical Ukrainian/Cyrillic documents, "
    "school homework, exams, tables, formulas, chemistry notation, teacher marks, or mixed handwriting/print. "
    "Read only visible characters. Do not complete from canonical or memorized text. "
    "Preserve old spelling and do not modernize. "
)

COMMON_OCR_RULES = (
    "Return only the transcription. No JSON, no Markdown, no explanation. "
    + DOCUMENT_CONTEXT_RULES
    + SPECIAL_TEXT_MARKER_RULES
    + "Preserve punctuation, line content, corrections, spelling mistakes, capitalization, digits, "
    "abbreviations, quotes, hyphens, line-final dashes, and visible spacing as much as possible. "
    "Do not translate, correct grammar, normalize spelling, expand abbreviations, summarize, "
    "or infer hidden/missing text."
)

PAGE_PROMPT = (
    "Extract every visible document region. Return only a compact JSON array. "
    "Each item must have keys bbox,type,text. bbox is [x1,y1,x2,y2] on a 0-1000 grid. "
    "type is one of handwritten,printed,formula,table,annotation,image,graph. "
    "Use empty text for image and graph. Preserve reading order. "
    "For every text field, transcribe only visible characters exactly; do not translate, summarize, "
    "normalize spelling, correct grammar, expand abbreviations, or infer hidden/missing text. "
    + PAGE_SPECIAL_TEXT_MARKER_RULES
    + "Preserve punctuation, corrections, spelling mistakes, capitalization, digits, quotes, hyphens, "
    "line-final dashes, and old Ukrainian/Cyrillic spelling. No Markdown, no explanation."
)

CROP_PROMPTS = {
    "handwritten": (
        "Transcribe the visible handwritten text exactly. "
        "Preserve punctuation, line content, corrections, and strikethrough markers. "
        + COMMON_OCR_RULES
    ),
    "printed": (
        "Transcribe the visible printed or typed text exactly. "
        "Preserve punctuation, line content, corrections, and strikethrough markers. "
        + COMMON_OCR_RULES
    ),
    "annotation": (
        "Read this short annotation, teacher mark, grade, correction, or numbering. "
        "Return only the exact visible text. "
        + COMMON_OCR_RULES
    ),
    "formula": (
        "Read this standalone math, logic, vector, matrix, determinant, set/relation, statistics, physics, "
        "or chemistry expression exactly as written. Return only formula text, using LaTeX when it is the "
        "clearest representation and plain Unicode when it better matches the handwriting. Do not wrap the "
        "answer in dollar signs. Preserve visible symbols, indices, superscripts, subscripts, arrows, fractions, "
        "matrix/determinant structure, punctuation, numbering, and strikethrough/correction markers. "
        + SPECIAL_TEXT_MARKER_RULES
        + "Do not solve, simplify, normalize, explain, or convert old notation into a different style."
    ),
    "table": (
        "Read this table region exactly. Return only pipe-separated table text. Use one output line per visual row "
        "and | between cells. Preserve empty cells with empty fields, for example A||C. Preserve row order, "
        "column order, multi-word cell text, wrapped cell text, numbers, units, punctuation, dashes, visible spelling "
        "mistakes, corrections, and strikethrough markers. "
        + SPECIAL_TEXT_MARKER_RULES
        + "Do not infer missing cells, rebalance columns, summarize, or explain."
    ),
    "image": "Return an empty string.",
    "graph": "Return an empty string.",
    "default": "Transcribe the visible content exactly. " + COMMON_OCR_RULES,
}

VALID_TYPES = {"handwritten", "printed", "formula", "table", "annotation", "image", "graph"}
SCORABLE_TYPES = {"formula"}
IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp", ".bmp"]

In [ ]:
def first_existing(paths):
    for item in paths:
        p = Path(item)
        if p.exists():
            return p
    return None


def find_model_id():
    for item in BASE_MODEL_CANDIDATES:
        if item.startswith("/") and Path(item).exists():
            return item
        if not item.startswith("/"):
            return item
    raise FileNotFoundError("No Qwen3-VL model path found. Add the Kaggle model or enable internet.")


def find_stage1_lora_dir():
    for item in STAGE1_LORA_CANDIDATES:
        p = Path(item)
        if (p / "adapter_config.json").exists():
            return p
    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = []
        for cfg_path in input_root.rglob("adapter_config.json"):
            parent = cfg_path.parent
            parent_text = str(parent).lower()
            if "silver" in parent_text or "stage1" in parent_text:
                matches.append(parent)
        if matches:
            return sorted(matches, key=lambda p: ("hybrid" not in str(p).lower(), len(str(p))))[0]
    searched = "\n".join(f"  - {p}" for p in STAGE1_LORA_CANDIDATES)
    raise FileNotFoundError(
        "No Stage 1 LoRA adapter_config.json found. Checked:\n" + searched
    )


def get_dataset_root():
    root = Path(DATASET_ROOT)
    train_meta = root / "train" / "metadata.jsonl"
    if not train_meta.exists():
        raise FileNotFoundError(
            f"DATASET_ROOT is not configured correctly: {root}. "
            "Expected train/metadata.jsonl under this path."
        )
    return root


def get_silver_split(root):
    if not USE_SILVER:
        return None
    silver_meta = root / SILVER_SPLIT / "metadata.jsonl"
    if not silver_meta.exists():
        raise FileNotFoundError(
            f"USE_SILVER=True but {silver_meta} does not exist. "
            "Set SILVER_SPLIT correctly or set USE_SILVER=False."
        )
    return SILVER_SPLIT


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def resolve_image_path(root, split, file_name):
    raw = Path(file_name)
    name = raw.name
    stem = raw.stem
    candidate_names = [name] + [stem + ext for ext in IMAGE_EXTENSIONS if stem + ext != name]
    candidates = [
        root / split / file_name,
        root / file_name,
    ]
    for candidate_name in candidate_names:
        candidates.extend([
            root / split / "images" / candidate_name,
            root / split / candidate_name,
        ])
    for p in candidates:
        if p.exists():
            return str(p)
    # Return the most likely path so the error message in PIL/qwen-vl-utils is useful.
    return str(root / split / "images" / candidate_names[0])


def clamp_box(box, w, h):
    if not isinstance(box, list) or len(box) != 4:
        return None
    try:
        x1, y1, x2, y2 = [float(v) for v in box]
    except Exception:
        return None
    x1, x2 = sorted((max(0, min(w, x1)), max(0, min(w, x2))))
    y1, y2 = sorted((max(0, min(h, y1)), max(0, min(h, y2))))
    if x2 - x1 < MIN_BOX_SIDE or y2 - y1 < MIN_BOX_SIDE:
        return None
    return [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]


def normalize_type(value):
    value = str(value or "handwritten").strip().lower()
    return value if value in VALID_TYPES else "handwritten"


def compact_page_regions(record):
    w = max(1, int(record.get("image_width") or 1))
    h = max(1, int(record.get("image_height") or 1))
    out = []
    for r in record.get("regions") or []:
        box = clamp_box(r.get("bbox"), w, h)
        if box is None:
            continue
        rtype = normalize_type(r.get("type"))
        text = r.get("text") or ""
        if rtype in {"image", "graph"}:
            text = ""
        nx1 = int(round(box[0] / w * 1000))
        ny1 = int(round(box[1] / h * 1000))
        nx2 = int(round(box[2] / w * 1000))
        ny2 = int(round(box[3] / h * 1000))
        out.append({"bbox": [nx1, ny1, nx2, ny2], "type": rtype, "text": str(text)})
    out.sort(key=lambda x: (x["bbox"][1], x["bbox"][0]))
    return out


def text_is_sane(text):
    if text is None:
        return False
    text = str(text).strip()
    if not text:
        return False
    if len(text) > 260:
        return False
    if re.search(r"(.)\1{10,}", text):
        return False
    bad = sum(ch in "\ufffd�" for ch in text)
    return bad == 0


def region_is_crop_candidate(region):
    rtype = normalize_type(region.get("type"))
    if rtype not in SCORABLE_TYPES:
        return False
    if str(region.get("language", "uk")).lower() == "other":
        return False
    if str(region.get("legibility", "legible")).lower() == "illegible":
        return False
    return text_is_sane(region.get("text"))


def stratified_split(records, val_ratio=0.12):
    by_source = defaultdict(list)
    for row in records:
        by_source[row.get("source", "unknown")].append(row)
    train_rows, val_rows = [], []
    rng = random.Random(SEED)
    for source, items in by_source.items():
        items = list(items)
        rng.shuffle(items)
        if len(items) <= 1:
            n_val = 0
        else:
            n_val = min(max(1, int(round(len(items) * val_ratio))), len(items) - 1)
        val_rows.extend(items[:n_val])
        train_rows.extend(items[n_val:])
    rng.shuffle(train_rows)
    rng.shuffle(val_rows)
    return train_rows, val_rows


In [ ]:
def make_page_sample(record, root, split):
    regions = compact_page_regions(record)
    if not regions or len(regions) > MAX_PAGE_REGIONS:
        return None
    answer = json.dumps(regions, ensure_ascii=False, separators=(",", ":"))
    if len(answer) > MAX_PAGE_ANSWER_CHARS:
        return None
    return {
        "task": "page_json",
        "image_path": resolve_image_path(root, split, record["file_name"]),
        "bbox": [0, 0, 0, 0],
        "region_type": "page",
        "prompt": PAGE_PROMPT,
        "answer": answer,
        "source": record.get("source", "unknown"),
    }


def make_crop_sample(record, region, root, split):
    w = max(1, int(record.get("image_width") or 1))
    h = max(1, int(record.get("image_height") or 1))
    box = clamp_box(region.get("bbox"), w, h)
    if box is None or not region_is_crop_candidate(region):
        return None
    rtype = normalize_type(region.get("type"))
    prompt = CROP_PROMPTS.get(rtype, CROP_PROMPTS["default"])
    return {
        "task": "crop_ocr",
        "image_path": resolve_image_path(root, "train", record["file_name"]),
        "bbox": box,
        "region_type": rtype,
        "prompt": prompt,
        "answer": str(region.get("text") or ""),
        "source": record.get("source", "unknown"),
    }


def balanced_take(samples, limit):
    if limit is None or len(samples) <= limit:
        return samples
    buckets = defaultdict(list)
    for s in samples:
        key = (s.get("source", "unknown"), s.get("region_type", s.get("task")))
        buckets[key].append(s)
    rng = random.Random(SEED)
    for items in buckets.values():
        rng.shuffle(items)
    selected = []
    keys = list(buckets.keys())
    while len(selected) < limit and keys:
        next_keys = []
        for key in keys:
            if buckets[key] and len(selected) < limit:
                selected.append(buckets[key].pop())
            if buckets[key]:
                next_keys.append(key)
        keys = next_keys
    rng.shuffle(selected)
    return selected


def build_samples(records, root, split, page_limit=None, crop_limit=None):
    rng = random.Random(SEED)
    rows = list(records)
    rng.shuffle(rows)
    page_samples, crop_samples = [], []
    for record in rows:
        page = make_page_sample(record, root, split)
        if page is not None:
            page_samples.append(page)
        for region in record.get("regions") or []:
            crop = make_crop_sample(record, region, root, split)
            if crop is not None:
                crop_samples.append(crop)
    page_samples = balanced_take(page_samples, page_limit)
    crop_samples = balanced_take(crop_samples, crop_limit)
    mixed = page_samples + crop_samples
    rng.shuffle(mixed)
    print(f"{split}: page={len(page_samples)} crop={len(crop_samples)} total={len(mixed)}")
    return mixed


root = get_dataset_root()
model_id = find_model_id()
stage1_lora_dir = find_stage1_lora_dir()

print("Dataset root:", root)
print("Stage 1 LoRA:", stage1_lora_dir)
print("Base model:", model_id)

gold_records = read_jsonl(root / "train" / "metadata.jsonl")
gold_train_records, gold_val_records = stratified_split(gold_records, VAL_RATIO)
print(f"Gold train pages={len(gold_train_records)} val pages={len(gold_val_records)}")

stage2_samples = build_samples(
    gold_train_records,
    root,
    "train",
    page_limit=GOLD_PAGE_LIMIT,
    crop_limit=GOLD_CROP_LIMIT,
)
if not stage2_samples:
    raise RuntimeError(f"No gold training samples were built. Check {root / 'train' / 'metadata.jsonl'}.")
stage2_ds = Dataset.from_list(stage2_samples)

val_jsonl = OUTPUT_DIR / "gold_validation_records.jsonl"
with open(val_jsonl, "w", encoding="utf-8") as f:
    for row in gold_val_records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("Saved validation records to", val_jsonl)


In [ ]:
from peft import PeftModel, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig, TrainerCallback
from trl import SFTConfig, SFTTrainer


def force_fp16_config(model):
    model.config.torch_dtype = torch.float16
    for attr in ("text_config", "vision_config"):
        cfg = getattr(model.config, attr, None)
        if cfg is not None:
            cfg.torch_dtype = torch.float16
            if hasattr(cfg, "dtype"):
                cfg.dtype = "float16"


processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

max_memory = {0: "14GiB"}
if torch.cuda.device_count() > 1:
    max_memory[1] = "14GiB"

base_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map=MODEL_DEVICE_MAP,
    max_memory=max_memory,
    quantization_config=quantization_config,
    dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
force_fp16_config(base_model)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
try:
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except TypeError:
    base_model.gradient_checkpointing_enable()

model = PeftModel.from_pretrained(base_model, str(stage1_lora_dir), is_trainable=True)
for _, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
model.print_trainable_parameters()


In [ ]:
def crop_with_padding(image_path, bbox, pad_ratio=CROP_PAD_RATIO):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        w, h = img.size
        x1, y1, x2, y2 = bbox
        pad = int(round(max(x2 - x1, y2 - y1) * pad_ratio))
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(w, x2 + pad)
        y2 = min(h, y2 + pad)
        return img.crop((x1, y1, x2, y2))


def build_messages(sample):
    if sample["task"] == "page_json":
        image_obj = sample["image_path"]
        max_pixels = MAX_PIXELS_PAGE
    else:
        image_obj = crop_with_padding(sample["image_path"], sample["bbox"])
        max_pixels = MAX_PIXELS_CROP
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_obj, "max_pixels": max_pixels},
                {"type": "text", "text": sample["prompt"]},
            ],
        },
        {"role": "assistant", "content": [{"type": "text", "text": sample["answer"]}]},
    ]


def encode_marker(tokenizer):
    try:
        return tokenizer.encode("<|im_start|>assistant\n", allowed_special="all", add_special_tokens=False)
    except TypeError:
        return tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)


ASSISTANT_MARKER = encode_marker(processor.tokenizer)


def data_collator(examples):
    messages_list = [build_messages(ex) for ex in examples]
    texts = [
        processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in messages_list
    ]
    image_inputs, video_inputs = process_vision_info(messages_list)
    batch = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100

    eos_id = processor.tokenizer.eos_token_id
    for i in range(labels.shape[0]):
        ids = batch["input_ids"][i].tolist()
        start = -1
        for j in range(0, len(ids) - len(ASSISTANT_MARKER) + 1):
            if ids[j:j + len(ASSISTANT_MARKER)] == ASSISTANT_MARKER:
                start = j + len(ASSISTANT_MARKER)
                break
        actual_len = int(batch["attention_mask"][i].sum().item())
        truncated = actual_len >= MAX_SEQ_LENGTH and (eos_id is None or ids[actual_len - 1] != eos_id)
        if start >= 0 and not truncated:
            labels[i, :start] = -100
        else:
            labels[i, :] = -100

    batch["labels"] = labels
    for key, value in list(batch.items()):
        if isinstance(value, torch.Tensor) and value.dtype == torch.float32:
            batch[key] = value.to(torch.float16)
    return batch


In [ ]:
class PrintProgressCallback(TrainerCallback):
    def __init__(self, name):
        self.name = name
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f"[{self.name}] start: max_steps={state.max_steps}, grad_accum={args.gradient_accumulation_steps}", flush=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        elapsed = time.time() - (self.start_time or time.time())
        step = max(1, state.global_step)
        eta = (elapsed / step) * max(0, state.max_steps - step)
        loss = logs.get("loss", logs.get("eval_loss", None))
        lr = logs.get("learning_rate", None)
        msg = f"[{self.name}] step {state.global_step}/{state.max_steps}"
        if loss is not None:
            msg += f" loss={loss:.4f}"
        if lr is not None:
            msg += f" lr={lr:.2e}"
        msg += f" elapsed={elapsed/60:.1f}m eta={eta/60:.1f}m"
        if torch.cuda.is_available():
            msg += " vram=" + ",".join(
                f"{i}:{torch.cuda.memory_allocated(i)/1024**3:.1f}GB"
                for i in range(torch.cuda.device_count())
            )
        print(msg, flush=True)


class OOMRecoverySFTTrainer(SFTTrainer):
    def training_step(self, model, inputs, num_items_in_batch=None):
        try:
            try:
                loss = super().training_step(model, inputs, num_items_in_batch=num_items_in_batch)
            except TypeError:
                loss = super().training_step(model, inputs)
            if self.args.device != loss.device:
                loss = loss.to(self.args.device)
            return loss
        except torch.cuda.OutOfMemoryError:
            print("OOM: skipping one batch after clearing cache.", flush=True)
            for p in model.parameters():
                p.grad = None
            torch.cuda.empty_cache()
            gc.collect()
            return torch.tensor(0.0, device=self.args.device)


training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "stage2_gold"),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=3e-5,
    max_steps=STAGE2_GOLD_MAX_STEPS,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=max(1, int(STAGE2_GOLD_MAX_STEPS * 0.03)),
    max_grad_norm=0.3,
    logging_steps=10,
    save_strategy="steps",
    save_steps=max(100, STAGE2_GOLD_MAX_STEPS // 2),
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    dataloader_pin_memory=False,
    dataloader_persistent_workers=DATALOADER_NUM_WORKERS > 0,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = OOMRecoverySFTTrainer(
    model=model,
    args=training_args,
    train_dataset=stage2_ds,
    data_collator=data_collator,
    callbacks=[PrintProgressCallback("gold")],
)
trainer.train()

final_dir = OUTPUT_DIR / "qwen3vl_rukopys_lora_final"
trainer.model.save_pretrained(final_dir)
processor.save_pretrained(final_dir)
(final_dir / "rukopys_prompt_config.json").write_text(
    json.dumps({
        "prompt_version": PROMPT_VERSION,
        "page_prompt": PAGE_PROMPT,
        "crop_prompts": CROP_PROMPTS,
        "max_pixels_page": MAX_PIXELS_PAGE,
        "max_pixels_crop": MAX_PIXELS_CROP,
        "base_model": str(model_id),
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Saved final gold LoRA adapter to", final_dir)


In [ ]:
# Optional quick sanity check on one validation image. This is not a leaderboard estimate.
RUN_QUICK_SANITY = True

if RUN_QUICK_SANITY and gold_val_records:
    model.eval()
    sample = gold_val_records[0]
    img_path = resolve_image_path(root, "train", sample["file_name"])
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path, "max_pixels": MAX_PIXELS_PAGE},
                {"type": "text", "text": PAGE_PROMPT},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
        out = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    print(processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0][:2000])
